# Notebook 04 — MT Baseline Training (Cascaded C0 stage 2)

**Scope:** Fine-tune locked pretrained Seq2Seq MT on gold parallel text:

`text_bahnar (reference) → text_vi (reference)`

Locked baseline (BARTPho syllable direct fine-tuning — **not** BARTBahnar / BV-BARTpho; no Bahnar continual pretraining):

- `MODEL_ID = vinai/bartpho-syllable`
- `MODEL_REVISION = 36eee8b4d648dd99da56462edcda3c5c97f7f3de`
- `EXPERIMENT_ID = mt_bartpho_syllable_v1`
- Best checkpoint: `eval_sacrebleu` (greater_is_better=True); chrF++ secondary

This notebook does **not**:
- continue Notebook 03 ASR / use XLS-R CTC / audio / ASR hypotheses
- open frozen RQ1 `G_test`
- train from random weights

**Stages:** `prepare` → `resume_test_a` → *(restart kernel)* → `resume_test_b` → `train` → `evaluate`

`READY_FOR_CASCADED_C0` only after independent `SUCCESS_MT_EVALUATE`.


In [ ]:
import os
from pathlib import Path

os.environ["BAHNAR_LOCAL_ROOT"] = "/tmp/bahnar-runtime"
os.environ["BAHNAR_DURABLE_ROOT"] = "/workspace/bahnar-s2tt-thesis"
os.environ["BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES"] = "32212254720"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

Path(os.environ["BAHNAR_LOCAL_ROOT"]).mkdir(parents=True, exist_ok=True)

print("BAHNAR_LOCAL_ROOT =", os.environ["BAHNAR_LOCAL_ROOT"])
print("BAHNAR_DURABLE_ROOT =", os.environ["BAHNAR_DURABLE_ROOT"])
print("BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES =", os.environ["BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES"])
print("HF_HUB_ENABLE_HF_TRANSFER =", os.environ["HF_HUB_ENABLE_HF_TRANSFER"])

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)



In [ ]:
%pip install -q "sacrebleu>=2.3,<3"



In [ ]:
# Bootstrap
import os, sys
from pathlib import Path
_NB_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
_PROJECT_ROOT = _NB_DIR.parent if _NB_DIR.name == "notebooks" else _NB_DIR
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
print("Project root:", _PROJECT_ROOT)



In [ ]:
# Imports
import json, traceback, uuid, os, platform
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import torch
from tqdm.auto import tqdm

from src.seed import set_seed
from src.data_utils import sha256_file, compute_uid_set_hash
from src.asr_utils import is_forbidden_test_path
from src.asr_full_train import (
    assert_no_frozen_test_access,
    durable_experiment_dir,
    ensure_experiment_fingerprint,
    make_durable_checkpoint_sync_callback,
    make_resume_proof_callback,
    new_session_token,
    resolve_durable_checkpoint_budget_bytes,
    resolve_resume_checkpoint,
    restore_experiment_checkpoints_from_durable,
    sync_experiment_checkpoints_to_durable,
    assert_checkpoint_complete_for_resume,
    assert_cross_session_resume,
    make_sequential_sampler_trainer_cls,
    make_uid_tracking_collator,
    make_uid_tracking_trainer_cls,
    plan_expected_resume_position,
    summarize_resume_proof,
    derive_resume_test_status_from_proof,
    write_checkpoint_fingerprint,
    list_step_checkpoints,
)
from src.mt_runtime_paths import (
    FULL_TRAIN_MARKER,
    PILOT_MARKER,
    RESUME_TEST_MARKER,
    SOURCE_FIELD,
    TARGET_FIELD,
    resolve_mt_runtime_paths,
)
from src.mt_contract import (
    LOCKED_EXPERIMENT_ID,
    LOCKED_GREATER_IS_BETTER,
    LOCKED_HARD_MAX_TRUNCATION_RATE,
    LOCKED_HARD_MAX_UNK_RATE,
    LOCKED_METRIC_FOR_BEST_MODEL,
    LOCKED_MODEL_ID,
    LOCKED_MODEL_REVISION,
    LOCKED_MT_MONITOR_SIZE,
    STATUS_FAILED,
    STATUS_MT_EVALUATE,
    STATUS_MT_PREPARE,
    STATUS_MT_RESUME_TEST,
    STATUS_MT_TRAINING,
    assert_locked_bartpho_baseline,
    assert_model_revision_pinned,
    assert_mt_training_contract_self_consistent,
    assert_mt_training_contracts_match,
    build_generation_config,
    build_mt_data_contract,
    build_mt_resume_test_contract,
    build_mt_training_contract,
    fingerprint_extra_from_mt_training_contract,
)
from src.mt_normalize import normalize_mt_text_v1, mt_normalization_version
from src.mt_prepare import (
    build_mt_exclusions,
    load_locked_mt_frames,
    load_mt_prepare_success,
    run_mt_prepare,
    verify_notebook02_locked_contract,
)
from src.mt_tokenize import (
    HARD_MAX_TRUNCATION_RATE,
    HARD_MAX_UNK_RATE,
    assert_tokenizer_compatible,
    audit_tokenizer_compatibility,
    load_mt_tokenizer,
    tokenizer_fingerprint,
)
from src.mt_dataset import MtTextDataset, make_seq2seq_collator
from src.mt_full_train import (
    RESUME_TEST_PHASE_A_STEPS,
    RESUME_TEST_PHASE_B_STEPS,
    RESUME_TEST_SUBSET_SIZE,
    assert_evaluate_config_consistency,
    assert_mt_checkpoint_namespace,
    assert_ready_for_mt_evaluate,
    assert_ready_for_mt_full_train,
    build_mt_resume_test_subset,
    build_mt_training_hparams,
    build_mt_validation_monitor_subset,
    compute_mt_seq2seq_metrics,
    derive_frozen_test_accessed,
    derive_mt_evaluate_status,
    derive_mt_training_status,
    derive_notebook04_handoff,
    derive_phase_a_reached_target,
    derive_started_from_base_or_same_experiment,
    derive_used_separate_experiment_dir,
    ensure_mt_full_train_fingerprint,
    evaluate_mt_predictions,
    generate_mt_predictions,
    load_durable_tokenizer_audit,
    load_mt_resume_test_contract,
    load_mt_seq2seq_model,
    load_mt_train_success,
    load_mt_training_contract,
    mt_experiment_dir,
    persist_durable_tokenizer_audit,
    persist_mt_monitor_manifest,
    resolve_latest_checkpoint_dir,
    resolve_mt_best_checkpoint_for_evaluate,
    write_mt_evaluate_summary,
    write_mt_full_train_fingerprint,
    commit_mt_resume_test_phase_b_durable,
    write_mt_resume_test_contract,
    write_mt_resume_test_fingerprint,
    write_mt_resume_test_summary,
    write_mt_train_summary,
    write_mt_training_contract,
)
from src.mt_export import export_notebook04_run
from src.metrics import mt_corpus_metrics, metrics_are_finite_values
print("imports ok")
assert HARD_MAX_UNK_RATE == LOCKED_HARD_MAX_UNK_RATE == 0.05
assert HARD_MAX_TRUNCATION_RATE == LOCKED_HARD_MAX_TRUNCATION_RATE == 0.05


## Configuration

In [ ]:
# ---- Run mode / stages ----------------------------------------------------
RUN_MODE = "pilot"   # "pilot" | "full"
FULL_STAGE = "prepare"  # prepare|resume_test_a|resume_test_b|train|evaluate
ALLOW_FULL_TRAINING = True
FULL_TRAINING_IMPLEMENTED = True

# ---- Locked BARTPho syllable baseline -------------------------------------
MODEL_ID = LOCKED_MODEL_ID
MODEL_REVISION = LOCKED_MODEL_REVISION
EXPERIMENT_ID = LOCKED_EXPERIMENT_ID
assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)

# ---- Data (locked NB01/02) ------------------------------------------------
DATASET_ID = "cuong06/Bahnar_Vietnamese"
EXPECTED_DATASET_REVISION = "3d88d3951b1a6e3388559b341cd7bd274879d696"
SEED = 42
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
METRIC_FOR_BEST_MODEL = LOCKED_METRIC_FOR_BEST_MODEL
GREATER_IS_BETTER = LOCKED_GREATER_IS_BETTER
MT_MONITOR_SIZE = LOCKED_MT_MONITOR_SIZE  # 512; training monitor only

# ---- Training hparams (centralized) ---------------------------------------
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
NUM_TRAIN_EPOCHS = 3.0
MAX_STEPS = -1
SAVE_STEPS = 500
EVAL_STEPS = 500
SAVE_TOTAL_LIMIT = 2
FP16 = True
BF16 = False
GRADIENT_CHECKPOINTING = True
GENERATION_MAX_LENGTH = 256  # shared train/eval generation policy
NUM_BEAMS = 4
PILOT_MAX_STEPS = 5
PILOT_SUBSET = 32

RUNTIME = resolve_mt_runtime_paths(project_root=_PROJECT_ROOT)
LOCAL_ROOT = RUNTIME.local_root
DURABLE_ROOT = RUNTIME.durable_root
LOCAL_CKPT_ROOT = RUNTIME.local_ckpt_root
FULL_STATE_DIR = RUNTIME.durable_state_root

PATHS = {
    "manifests": _PROJECT_ROOT / "data" / "manifests",
    "train_manifest": _PROJECT_ROOT / "data" / "manifests" / "rq1_train.csv",
    "val_manifest": _PROJECT_ROOT / "data" / "manifests" / "rq1_validation.csv",
    "train_excl": _PROJECT_ROOT / "data" / "audit" / "notebook02_train_contamination_exclusions.csv",
    "val_excl": _PROJECT_ROOT / "data" / "audit" / "notebook02_validation_contamination_exclusions.csv",
    "contract_nb02": _PROJECT_ROOT / "results" / "notebook02_clean_split_contract.json",
    "artifacts_root": _PROJECT_ROOT / "artifacts" / "notebook04",
    "results_root": _PROJECT_ROOT / "results" / "notebook04",
}
RUN_ID = str(uuid.uuid4())
PATHS["artifacts"] = PATHS["artifacts_root"] / RUN_ID
PATHS["results"] = PATHS["results_root"] / RUN_ID
for k in ("artifacts", "results"):
    PATHS[k].mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"RUN_ID={RUN_ID}  RUN_MODE={RUN_MODE}  FULL_STAGE={FULL_STAGE}  device={device}")
print(f"MODEL_ID={MODEL_ID!r}  MODEL_REVISION={MODEL_REVISION!r}")
print(f"monitor_size={MT_MONITOR_SIZE} unk_gate={HARD_MAX_UNK_RATE} trunc_gate={HARD_MAX_TRUNCATION_RATE}")
print(RUNTIME.as_dict())


In [ ]:
# Pipeline state
pipeline_error = None
pipeline_error_stage = None
can_continue = True
full_status = None
STATUS = "RUNNING"
OPENED_PATHS: List[str] = []
LOADED_SPLITS: List[str] = []

def fail_stage(stage, exc):
    global pipeline_error, pipeline_error_stage, can_continue, STATUS
    if pipeline_error is None:
        pipeline_error = {
            "exception_type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
        }
        pipeline_error_stage = stage
        can_continue = False
        STATUS = STATUS_FAILED
    print(f"[STAGE FAILED: {stage}] {type(exc).__name__}: {exc}")

print("pipeline state initialized")



## Frozen-test guard + NB02 contract

In [ ]:
if can_continue:
    try:
        nb02_verify = verify_notebook02_locked_contract(
            contract_path=PATHS["contract_nb02"],
            train_manifest=PATHS["train_manifest"],
            validation_manifest=PATHS["val_manifest"],
            train_exclusion=PATHS["train_excl"],
            validation_exclusion=PATHS["val_excl"],
            expected_dataset_revision=EXPECTED_DATASET_REVISION,
            opened_paths=OPENED_PATHS,
        )
        print("NB02 locked-contract verification PASS", nb02_verify)
    except Exception as e:
        fail_stage("prereq", e)


## Tokenizer load + compatibility probe

In [ ]:
tokenizer = None
tok_fp = None
tok_audit = None
if can_continue:
    try:
        # Full mode (all stages including prepare) and pilot: revision must be pinned.
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        tokenizer = load_mt_tokenizer(MODEL_ID, MODEL_REVISION)
        tok_fp = tokenizer_fingerprint(tokenizer)

        # Non-prepare full stages reuse durable audit after contract resolve (cell 18).
        defer_durable_audit = RUN_MODE == "full" and FULL_STAGE != "prepare"
        if defer_durable_audit:
            print("tokenizer fingerprint:", tok_fp)
            print("tokenizer audit deferred to durable contract state")
        else:
            # Pilot / prepare: audit on MT-eligible text after NB02 clean + MT exclusions.
            loaded = load_locked_mt_frames(
                train_manifest=PATHS["train_manifest"],
                validation_manifest=PATHS["val_manifest"],
                train_exclusion=PATHS["train_excl"],
                validation_exclusion=PATHS["val_excl"],
                opened_paths=OPENED_PATHS,
            )
            train_elig_audit, val_elig_audit, _, excl_rep = build_mt_exclusions(
                loaded["train"], loaded["validation"]
            )
            LOADED_SPLITS.extend(["train", "validation"])
            src_col = f"{SOURCE_FIELD}_norm"
            tgt_col = f"{TARGET_FIELD}_norm"
            sources = (
                train_elig_audit[src_col].astype(str).tolist()
                + val_elig_audit[src_col].astype(str).tolist()
            )
            targets = (
                train_elig_audit[tgt_col].astype(str).tolist()
                + val_elig_audit[tgt_col].astype(str).tolist()
            )
            tok_audit = audit_tokenizer_compatibility(
                tokenizer,
                sources=sources,
                targets=targets,
                max_source_length=MAX_SOURCE_LENGTH,
                max_target_length=MAX_TARGET_LENGTH,
            )
            tok_audit["tokenizer_fingerprint"] = tok_fp
            tok_audit["max_source_length"] = int(MAX_SOURCE_LENGTH)
            tok_audit["max_target_length"] = int(MAX_TARGET_LENGTH)
            tok_audit["audit_scope"] = {
                "policy": "full_eligible_train_and_validation",
                "train_eligible": int(len(train_elig_audit)),
                "validation_eligible": int(len(val_elig_audit)),
                "n_pairs": int(len(sources)),
                "exclusion_report_keys": list(excl_rep.keys()),
            }
            assert_tokenizer_compatible(tok_audit)
            (PATHS["artifacts"] / "tokenizer_audit.json").write_text(
                json.dumps(tok_audit, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            print("tokenizer fingerprint:", tok_fp)
            print(
                "tokenizer audit passed:",
                tok_audit["passed"],
                "n=",
                tok_audit["n"],
                "src_unk_rate=",
                tok_audit["source"]["unk_rate"],
                "tgt_unk_rate=",
                tok_audit["target"]["unk_rate"],
            )
    except Exception as e:
        fail_stage("tokenizer_probe", e)


## FULL_STAGE = prepare

In [ ]:
prepare_out = None
mt_contract = None
if can_continue and RUN_MODE == "full" and FULL_STAGE == "prepare":
    try:
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        assert tok_fp, "tokenizer fingerprint required before prepare"
        scratch = RUNTIME.durable_state_root / "_prepare_scratch" / RUN_ID
        prepare_out = run_mt_prepare(
            state_dir=scratch,
            train_manifest=PATHS["train_manifest"],
            validation_manifest=PATHS["val_manifest"],
            train_exclusion=PATHS["train_excl"],
            validation_exclusion=PATHS["val_excl"],
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            opened_paths=OPENED_PATHS,
        )
        mt_contract = prepare_out["contract"]
        FULL_STATE_DIR = RUNTIME.prepare_state_dir(mt_contract["contract_hash"])
        FULL_STATE_DIR.mkdir(parents=True, exist_ok=True)
        prepare_out = run_mt_prepare(
            state_dir=FULL_STATE_DIR,
            train_manifest=PATHS["train_manifest"],
            validation_manifest=PATHS["val_manifest"],
            train_exclusion=PATHS["train_excl"],
            validation_exclusion=PATHS["val_excl"],
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            opened_paths=OPENED_PATHS,
        )
        mt_contract = prepare_out["contract"]
        # Persist durable tokenizer audit into contract state (not only RUN_ID artifacts)
        assert tok_audit is not None, "tokenizer audit required before durable persist"
        persist_durable_tokenizer_audit(
            FULL_STATE_DIR,
            tok_audit,
            tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            mt_data_contract_hash=str(mt_contract["contract_hash"]),
            train_uid_set_hash=str(mt_contract.get("train_uid_set_hash")),
            validation_uid_set_hash=str(mt_contract.get("validation_uid_set_hash")),
        )
        # Cleanup scratch after successful commit
        import shutil
        if scratch.exists():
            shutil.rmtree(scratch, ignore_errors=True)
        assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
        frozen_flag = derive_frozen_test_accessed(OPENED_PATHS, LOADED_SPLITS)
        full_status = STATUS_MT_PREPARE
        print(json.dumps({
            "status": full_status,
            "contract_hash": mt_contract["contract_hash"],
            "train_eligible": prepare_out["summary"]["train_eligible"],
            "validation_eligible": prepare_out["summary"]["validation_eligible"],
            "frozen_test_accessed": frozen_flag,
            "full_state_dir": str(FULL_STATE_DIR),
        }, indent=2))
    except Exception as e:
        fail_stage("mt_prepare", e)


## Pilot smoke (optional)

In [ ]:
if can_continue and RUN_MODE == "pilot":
    try:
        assert tokenizer is not None
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        # Deterministic eligible subset (same exclusion logic as prepare; no raw first-N)
        loaded = load_locked_mt_frames(
            train_manifest=PATHS["train_manifest"],
            validation_manifest=PATHS["val_manifest"],
            train_exclusion=PATHS["train_excl"],
            validation_exclusion=PATHS["val_excl"],
            opened_paths=OPENED_PATHS,
        )
        train_elig, _, _, _ = build_mt_exclusions(loaded["train"], loaded["validation"])
        LOADED_SPLITS.append("train")
        pdf = build_mt_resume_test_subset(train_elig, n_samples=min(PILOT_SUBSET, len(train_elig)), seed=SEED)
        assert len(pdf) >= 2, "pilot needs >=2 eligible text pairs"

        model = load_mt_seq2seq_model(MODEL_ID, MODEL_REVISION)
        model.to(device)
        model.train()
        ds = MtTextDataset(pdf, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
        collator = make_seq2seq_collator(tokenizer, model)
        batch = collator([{k: v for k, v in ds[i].items() if k != "record_uid"} for i in range(min(2, len(ds)))])
        batch = {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}

        named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
        probe_name, probe_param = named[0]
        before = probe_param.detach().float().clone()
        out = model(**batch)
        loss = out.loss
        assert torch.isfinite(loss), f"non-finite loss: {loss}"
        opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        delta = (probe_param.detach().float() - before).abs().sum().item()
        assert delta > 0, f"optimizer.step() did not change weights ({probe_name})"

        model.eval()
        with torch.no_grad():
            gen = model.generate(
                batch["input_ids"], attention_mask=batch["attention_mask"],
                max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            )
            decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        refs = [normalize_mt_text_v1(pdf.iloc[i][f"{TARGET_FIELD}_norm" if f"{TARGET_FIELD}_norm" in pdf.columns else TARGET_FIELD]) for i in range(min(2, len(pdf)))]
        m = mt_corpus_metrics(decoded[: len(refs)], refs)
        assert metrics_are_finite_values(m["sacrebleu"], m["chrfpp"]), m

        pilot_dir = mt_experiment_dir(LOCAL_CKPT_ROOT, kind=PILOT_MARKER, experiment_id=EXPERIMENT_ID)
        ckpt_path = pilot_dir / "checkpoint-pilot"
        pilot_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        write_checkpoint_fingerprint(pilot_dir, experiment_id=EXPERIMENT_ID, kind=PILOT_MARKER, global_step=0)

        from transformers import AutoModelForSeq2SeqLM
        reloaded = AutoModelForSeq2SeqLM.from_pretrained(str(ckpt_path)).to(device)
        reloaded.eval()
        with torch.no_grad():
            gen2 = reloaded.generate(
                batch["input_ids"], attention_mask=batch["attention_mask"],
                max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            )
            _ = tokenizer.batch_decode(gen2, skip_special_tokens=True)
        assert_mt_checkpoint_namespace(ckpt_path, allowed_kind=PILOT_MARKER)
        assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
        full_status = "SUCCESS_MT_PILOT"
        print("PILOT OK", {"loss": float(loss.detach().cpu()), "weight_delta": delta, "sacrebleu": m["sacrebleu"], "chrfpp": m["chrfpp"]})
    except Exception as e:
        fail_stage("mt_pilot", e)


## Resume test A/B + Train + Evaluate (orchestration stubs calling helpers)

In [ ]:
# Resolve contract-scoped state for non-prepare full stages
mt_train_contract = None
if can_continue and RUN_MODE == "full" and FULL_STAGE != "prepare":
    try:
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        assert tok_fp, "tokenizer fingerprint required"
        loaded = load_locked_mt_frames(
            train_manifest=PATHS["train_manifest"],
            validation_manifest=PATHS["val_manifest"],
            train_exclusion=PATHS["train_excl"],
            validation_exclusion=PATHS["val_excl"],
            opened_paths=OPENED_PATHS,
        )
        tr, va, _, _ = build_mt_exclusions(loaded["train"], loaded["validation"])
        mt_contract = build_mt_data_contract(
            dataset_id=DATASET_ID,
            dataset_revision=EXPECTED_DATASET_REVISION,
            train_manifest_content_hash=loaded["train_manifest_content_hash"],
            validation_manifest_content_hash=loaded["validation_manifest_content_hash"],
            train_uid_set_hash=compute_uid_set_hash(tr),
            validation_uid_set_hash=compute_uid_set_hash(va),
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
        )
        FULL_STATE_DIR = RUNTIME.prepare_state_dir(mt_contract["contract_hash"])
        print("FULL_STATE_DIR", FULL_STATE_DIR)
        load_mt_prepare_success(FULL_STATE_DIR, expected_contract=mt_contract)
        # Reuse durable tokenizer audit (no full re-tokenize of train+validation)
        tok_audit = load_durable_tokenizer_audit(
            FULL_STATE_DIR,
            tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            mt_data_contract_hash=str(mt_contract["contract_hash"]),
            train_uid_set_hash=str(mt_contract.get("train_uid_set_hash")),
            validation_uid_set_hash=str(mt_contract.get("validation_uid_set_hash")),
        )
        assert_tokenizer_compatible(tok_audit)
        (PATHS["artifacts"] / "tokenizer_audit.json").write_text(
            json.dumps(tok_audit, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print("durable tokenizer audit reused:", tok_audit.get("audit_sha256"))
    except Exception as e:
        fail_stage("resolve_contract_state", e)


In [ ]:
# resume_test_a / resume_test_b
if can_continue and RUN_MODE == "full" and FULL_STAGE in {"resume_test_a", "resume_test_b"}:
    try:
        from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
        assert_model_revision_pinned(MODEL_ID, MODEL_REVISION)
        train_elig = pd.read_csv(FULL_STATE_DIR / "mt_train_eligible.csv")
        LOADED_SPLITS.append("train")
        subset = build_mt_resume_test_subset(train_elig, n_samples=RESUME_TEST_SUBSET_SIZE, seed=SEED)
        rt_dir = mt_experiment_dir(LOCAL_CKPT_ROOT, kind=RESUME_TEST_MARKER, experiment_id=EXPERIMENT_ID)
        durable_rt = durable_experiment_dir(FULL_STATE_DIR, EXPERIMENT_ID, kind=RESUME_TEST_MARKER)
        tokenizer = load_mt_tokenizer(MODEL_ID, MODEL_REVISION)
        tok_fp = tokenizer_fingerprint(tokenizer)
        resume_contract = build_mt_resume_test_contract(
            mt_data_contract_hash=str(mt_contract["contract_hash"]),
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp,
            subset_uid_set_hash=compute_uid_set_hash(subset),
            seed=SEED,
            subset_size=RESUME_TEST_SUBSET_SIZE,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=LEARNING_RATE,
            phase_a_target_steps=RESUME_TEST_PHASE_A_STEPS,
            phase_b_target_steps=RESUME_TEST_PHASE_B_STEPS,
        )
        ds = MtTextDataset(subset, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
        budget = resolve_durable_checkpoint_budget_bytes(None, required=True)

        if FULL_STAGE == "resume_test_a":
            model = load_mt_seq2seq_model(MODEL_ID, MODEL_REVISION)
            session = new_session_token()
            write_mt_resume_test_contract(FULL_STATE_DIR, resume_contract)
            # Write real fingerprint BEFORE train/sync (empty-dir ensure is not enough)
            write_mt_resume_test_fingerprint(
                rt_dir,
                experiment_id=EXPERIMENT_ID,
                mt_data_contract=mt_contract,
                model_id=MODEL_ID,
                model_revision=MODEL_REVISION,
                tokenizer_fingerprint=tok_fp,
                global_step=0,
            )
            assert (Path(rt_dir) / "full_experiment_fingerprint.json").is_file()
            args = Seq2SeqTrainingArguments(
                output_dir=str(rt_dir),
                max_steps=RESUME_TEST_PHASE_A_STEPS,
                per_device_train_batch_size=1,
                gradient_accumulation_steps=8,
                learning_rate=LEARNING_RATE,
                save_steps=RESUME_TEST_PHASE_A_STEPS,
                save_total_limit=2,
                logging_steps=10,
                report_to=[],
                fp16=FP16 and device == "cuda",
                predict_with_generate=False,
                remove_unused_columns=True,  # strip record_uid before DataCollatorForSeq2Seq
            )
            collator = make_seq2seq_collator(tokenizer, model)
            TrainerCls = make_sequential_sampler_trainer_cls(Seq2SeqTrainer)
            trainer = TrainerCls(model=model, args=args, train_dataset=ds, data_collator=collator, processing_class=tokenizer)
            trainer.train()
            ckpt = rt_dir / f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            assert_checkpoint_complete_for_resume(ckpt)
            write_mt_resume_test_fingerprint(
                rt_dir,
                experiment_id=EXPERIMENT_ID,
                mt_data_contract=mt_contract,
                model_id=MODEL_ID,
                model_revision=MODEL_REVISION,
                tokenizer_fingerprint=tok_fp,
                global_step=RESUME_TEST_PHASE_A_STEPS,
            )
            sync_experiment_checkpoints_to_durable(
                rt_dir, FULL_STATE_DIR, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
                require_durable=True, save_total_limit=2, budget_bytes=budget,
            )
            payload = {
                "status": "PHASE_A_COMPLETE",
                "session": session,
                "global_step": RESUME_TEST_PHASE_A_STEPS,
                "contract_hash": mt_contract["contract_hash"],
                "checkpoint": str(ckpt),
            }
            (FULL_STATE_DIR / "mt_resume_test_phase_a.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
            assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
            full_status = "SUCCESS_MT_RESUME_TEST_PHASE_A"
            print("Phase A durable — RESTART runtime, set FULL_STAGE='resume_test_b'")
        else:
            # Phase B: verify resume-test contract exact match before resume
            load_mt_resume_test_contract(FULL_STATE_DIR, expected=resume_contract)
            phase_a = json.loads((FULL_STATE_DIR / "mt_resume_test_phase_a.json").read_text(encoding="utf-8"))
            session = new_session_token()
            cross = assert_cross_session_resume(phase_a, current_session=session)
            restore_experiment_checkpoints_from_durable(
                rt_dir, FULL_STATE_DIR, experiment_id=EXPERIMENT_ID, kind=RESUME_TEST_MARKER,
            )
            ckpt_a = rt_dir / f"checkpoint-{RESUME_TEST_PHASE_A_STEPS}"
            assert_checkpoint_complete_for_resume(ckpt_a)
            phase_a_ok = derive_phase_a_reached_target(
                phase_a, target_steps=RESUME_TEST_PHASE_A_STEPS, checkpoint_path=ckpt_a
            )
            used_separate = derive_used_separate_experiment_dir(rt_dir, allowed_kind=RESUME_TEST_MARKER)
            model = AutoModelForSeq2SeqLM.from_pretrained(str(ckpt_a))
            args = Seq2SeqTrainingArguments(
                output_dir=str(rt_dir),
                max_steps=RESUME_TEST_PHASE_B_STEPS,
                per_device_train_batch_size=1,
                gradient_accumulation_steps=8,
                learning_rate=LEARNING_RATE,
                save_steps=RESUME_TEST_PHASE_B_STEPS,
                save_total_limit=2,
                logging_steps=10,
                report_to=[],
                fp16=FP16 and device == "cuda",
                remove_unused_columns=False,  # keep record_uid for UID-tracking collator
            )
            position_sink: Dict[str, Any] = {}
            tracking_collator = make_uid_tracking_collator(make_seq2seq_collator(tokenizer, model), position_sink)
            TrackingTrainer = make_uid_tracking_trainer_cls(
                make_sequential_sampler_trainer_cls(Seq2SeqTrainer), position_sink
            )
            proof_cb = make_resume_proof_callback(
                checkpoint_dir=ckpt_a,
                expected_resume_step=RESUME_TEST_PHASE_A_STEPS,
                sink=position_sink,
                gradient_accumulation_steps=8,
            )
            trainer = TrackingTrainer(
                model=model, args=args, train_dataset=ds,
                data_collator=tracking_collator, processing_class=tokenizer,
                callbacks=[proof_cb],
            )
            expected = plan_expected_resume_position(
                subset["record_uid"].astype(str).tolist(),
                resume_step=RESUME_TEST_PHASE_A_STEPS,
                per_device_train_batch_size=1,
                gradient_accumulation_steps=8,
            )
            trainer.train(resume_from_checkpoint=str(ckpt_a))
            proof = summarize_resume_proof(position_sink, expected_first_uid=expected.get("expected_first_uid"))
            proof["final_global_step"] = int(getattr(trainer.state, "global_step", 0))
            verdict = derive_resume_test_status_from_proof(
                proof,
                phase_a_reached_target=phase_a_ok,
                cross_session=cross,
                used_separate_experiment_dir=used_separate,
                expected_phase_b_steps=RESUME_TEST_PHASE_B_STEPS,
            )
            status = STATUS_MT_RESUME_TEST if verdict["status"] == "SUCCESS_FULL_RESUME_TEST" else STATUS_FAILED
            assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
            frozen_flag = derive_frozen_test_accessed(OPENED_PATHS, LOADED_SPLITS)
            # Fail-closed: never persist SUCCESS before durable Phase B commit completes.
            if status != STATUS_MT_RESUME_TEST:
                write_mt_resume_test_summary(FULL_STATE_DIR, {
                    "status": STATUS_FAILED,
                    "contract_hash": mt_contract["contract_hash"],
                    "experiment_id": EXPERIMENT_ID,
                    "checks": verdict.get("checks"),
                    "failed_checks": verdict.get("failed_checks"),
                    "proof": proof,
                    "cross_session": cross,
                    "phase_a_reached_target": phase_a_ok,
                    "used_separate_experiment_dir": used_separate,
                    "frozen_test_accessed": frozen_flag,
                    "durable_commit_ok": False,
                })
                full_status = STATUS_FAILED
                print(json.dumps({"status": STATUS_FAILED, "failed_checks": verdict.get("failed_checks")}, indent=2))
                raise RuntimeError(f"MT resume_test failed: {verdict.get('failed_checks')}")

            write_mt_resume_test_fingerprint(
                rt_dir,
                experiment_id=EXPERIMENT_ID,
                mt_data_contract=mt_contract,
                model_id=MODEL_ID,
                model_revision=MODEL_REVISION,
                tokenizer_fingerprint=tok_fp,
                global_step=RESUME_TEST_PHASE_B_STEPS,
            )
            try:
                durable_commit = commit_mt_resume_test_phase_b_durable(
                    rt_dir,
                    FULL_STATE_DIR,
                    experiment_id=EXPERIMENT_ID,
                    phase_a_steps=RESUME_TEST_PHASE_A_STEPS,
                    phase_b_steps=RESUME_TEST_PHASE_B_STEPS,
                    budget_bytes=budget,
                    save_total_limit=2,
                )
            except Exception as sync_err:
                write_mt_resume_test_summary(FULL_STATE_DIR, {
                    "status": STATUS_FAILED,
                    "contract_hash": mt_contract["contract_hash"],
                    "experiment_id": EXPERIMENT_ID,
                    "checks": verdict.get("checks"),
                    "failed_checks": list(verdict.get("failed_checks") or []) + ["durable_commit"],
                    "proof": proof,
                    "cross_session": cross,
                    "phase_a_reached_target": phase_a_ok,
                    "used_separate_experiment_dir": used_separate,
                    "frozen_test_accessed": frozen_flag,
                    "durable_commit_ok": False,
                    "durable_commit_error": str(sync_err),
                })
                full_status = STATUS_FAILED
                raise

            write_mt_resume_test_summary(FULL_STATE_DIR, {
                "status": STATUS_MT_RESUME_TEST,
                "contract_hash": mt_contract["contract_hash"],
                "experiment_id": EXPERIMENT_ID,
                "checks": verdict.get("checks"),
                "failed_checks": verdict.get("failed_checks"),
                "proof": proof,
                "cross_session": cross,
                "phase_a_reached_target": phase_a_ok,
                "used_separate_experiment_dir": used_separate,
                "frozen_test_accessed": frozen_flag,
                "durable_commit_ok": True,
                "durable_commit_mode": durable_commit.get("mode"),
                "attempt_id": durable_commit.get("attempt_id"),
                "phase_b_digest": durable_commit.get("phase_b_digest"),
                "phase_b_durable_relpath": durable_commit.get("phase_b_durable_relpath"),
            })
            full_status = STATUS_MT_RESUME_TEST
            print(json.dumps({
                "status": STATUS_MT_RESUME_TEST,
                "failed_checks": verdict.get("failed_checks"),
                "durable_commit": {
                    "mode": durable_commit.get("mode"),
                    "attempt_id": durable_commit.get("attempt_id"),
                    "phase_b_durable_relpath": durable_commit.get("phase_b_durable_relpath"),
                },
            }, indent=2))
    except Exception as e:
        fail_stage("mt_resume_test", e)


In [ ]:
# FULL_STAGE=train
if can_continue and RUN_MODE == "full" and FULL_STAGE == "train":
    try:
        if not ALLOW_FULL_TRAINING:
            raise RuntimeError("ALLOW_FULL_TRAINING is False — refusing full MT train")
        from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
        assert_ready_for_mt_full_train(FULL_STATE_DIR, contract=mt_contract)
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        train_elig = pd.read_csv(FULL_STATE_DIR / "mt_train_eligible.csv")
        val_elig = pd.read_csv(FULL_STATE_DIR / "mt_validation_eligible.csv")
        LOADED_SPLITS.extend(["train", "validation"])
        tokenizer = load_mt_tokenizer(MODEL_ID, MODEL_REVISION)
        tok_fp = tokenizer_fingerprint(tokenizer)
        model = load_mt_seq2seq_model(MODEL_ID, MODEL_REVISION)
        loaded_from_pinned_base = True
        model.to(device)

        # Fixed monitor subset for mid-training eval (NOT final validation)
        monitor_df = build_mt_validation_monitor_subset(val_elig, n_samples=MT_MONITOR_SIZE)
        monitor_manifest = persist_mt_monitor_manifest(
            FULL_STATE_DIR, monitor_df, n_samples=MT_MONITOR_SIZE
        )
        (PATHS["artifacts"] / "mt_validation_monitor_manifest.json").write_text(
            json.dumps(monitor_manifest, indent=2), encoding="utf-8"
        )

        train_ds = MtTextDataset(train_elig, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
        monitor_ds = MtTextDataset(monitor_df, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)

        mt_train_contract = build_mt_training_contract(
            mt_data_contract_hash=str(mt_contract["contract_hash"]),
            train_uid_set_hash=str(mt_contract["train_uid_set_hash"]),
            validation_uid_set_hash=str(mt_contract["validation_uid_set_hash"]),
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp,
            experiment_id=EXPERIMENT_ID,
            seed=SEED,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            max_steps=MAX_STEPS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            fp16=FP16,
            bf16=BF16,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            save_steps=SAVE_STEPS,
            eval_steps=EVAL_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            generation_max_length=GENERATION_MAX_LENGTH,
            num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL,
            greater_is_better=GREATER_IS_BETTER,
            monitor_size=MT_MONITOR_SIZE,
            monitor_uid_set_hash=str(monitor_manifest["uid_set_hash"]),
        )
        write_mt_training_contract(FULL_STATE_DIR, mt_train_contract)

        exp_dir = mt_experiment_dir(LOCAL_CKPT_ROOT, kind=FULL_TRAIN_MARKER, experiment_id=EXPERIMENT_ID)
        durable_exp = durable_experiment_dir(FULL_STATE_DIR, EXPERIMENT_ID, kind=FULL_TRAIN_MARKER)
        # Validate existing fingerprint identity BEFORE any stamp / restore
        ensure_mt_full_train_fingerprint(exp_dir, train_contract=mt_train_contract, global_step=0)
        expected_fp = fingerprint_extra_from_mt_training_contract(mt_train_contract)
        restored = restore_experiment_checkpoints_from_durable(
            exp_dir, FULL_STATE_DIR, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            expected_contract={
                "experiment_id": EXPERIMENT_ID,
                "hparams": expected_fp["hparams"],
                "mt_training_contract_hash": mt_train_contract["mt_training_contract_hash"],
                "contract_hash": mt_train_contract["mt_training_contract_hash"],
            },
        )
        print("durable restore:", restored)
        # Re-validate after restore (checkpoints may now exist)
        ensure_mt_full_train_fingerprint(exp_dir, train_contract=mt_train_contract, global_step=0)
        resume_ckpt = resolve_resume_checkpoint(
            resume_policy="auto", experiment_dir=exp_dir, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
        )
        print("resume_from_checkpoint:", resume_ckpt)
        started_ok = derive_started_from_base_or_same_experiment(
            resume_ckpt=resume_ckpt,
            experiment_id=EXPERIMENT_ID,
            train_contract=mt_train_contract,
            local_experiment_dir=exp_dir,
            loaded_from_pinned_base=loaded_from_pinned_base,
        )

        gen_cfg = build_generation_config(
            model_id=MODEL_ID, model_revision=MODEL_REVISION, tokenizer_fingerprint=tok_fp,
            max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH,
            generation_max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL, greater_is_better=GREATER_IS_BETTER,
            normalization_version=mt_normalization_version(),
        )
        durable_sync_cb = make_durable_checkpoint_sync_callback(
            local_experiment_dir=exp_dir, full_state_dir=FULL_STATE_DIR,
            experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            training_contract={
                "experiment_id": EXPERIMENT_ID,
                "hparams": expected_fp["hparams"],
                "mt_training_contract_hash": mt_train_contract["mt_training_contract_hash"],
                "contract_hash": mt_train_contract["mt_training_contract_hash"],
                **expected_fp,
            },
            save_total_limit=SAVE_TOTAL_LIMIT,
            budget_bytes=resolve_durable_checkpoint_budget_bytes(None, required=True),
        )
        hparams = build_mt_training_hparams(
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            seed=SEED,
            max_steps=MAX_STEPS,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            save_steps=SAVE_STEPS,
            eval_steps=EVAL_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            fp16=FP16,
            bf16=BF16,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            generation_max_length=GENERATION_MAX_LENGTH,
            num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL,
            greater_is_better=GREATER_IS_BETTER,
        )

        def compute_metrics(eval_preds):
            # Replace -100 in both preds and labels before decode (Trainer may pad preds with -100).
            return compute_mt_seq2seq_metrics(eval_preds, tokenizer=tokenizer)

        args = Seq2SeqTrainingArguments(
            output_dir=str(exp_dir),
            num_train_epochs=NUM_TRAIN_EPOCHS,
            max_steps=MAX_STEPS if MAX_STEPS and MAX_STEPS > 0 else -1,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            load_best_model_at_end=True,
            metric_for_best_model=METRIC_FOR_BEST_MODEL.replace("eval_", ""),
            greater_is_better=GREATER_IS_BETTER,
            predict_with_generate=True,
            generation_max_length=GENERATION_MAX_LENGTH,
            generation_num_beams=NUM_BEAMS,
            fp16=FP16 and device == "cuda",
            bf16=BF16,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            report_to=[],
            seed=SEED,
            remove_unused_columns=True,  # strip record_uid before DataCollatorForSeq2Seq
        )
        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=monitor_ds,  # monitor subset only
            data_collator=make_seq2seq_collator(tokenizer, model),
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            callbacks=[durable_sync_cb],
        )
        train_result = trainer.train(resume_from_checkpoint=str(resume_ckpt) if resume_ckpt else None)
        metrics = trainer.evaluate()  # still monitor subset
        best = getattr(trainer.state, "best_model_checkpoint", None)
        best_name = Path(str(best)).name if best else None
        latest_path = resolve_latest_checkpoint_dir(exp_dir)
        latest_name = latest_path.name if latest_path else None

        # Final durable sync MUST pass best_checkpoint_name (preserve best != latest)
        sync_experiment_checkpoints_to_durable(
            exp_dir, FULL_STATE_DIR, experiment_id=EXPERIMENT_ID, kind=FULL_TRAIN_MARKER,
            require_durable=True, save_total_limit=SAVE_TOTAL_LIMIT,
            best_checkpoint_name=best_name,
            budget_bytes=resolve_durable_checkpoint_budget_bytes(None, required=True),
        )
        ensure_mt_full_train_fingerprint(
            exp_dir, train_contract=mt_train_contract,
            global_step=int(trainer.state.global_step),
            update_global_step=True,
        )

        assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
        frozen_flag = derive_frozen_test_accessed(OPENED_PATHS, LOADED_SPLITS)
        status = derive_mt_training_status(
            global_step=int(trainer.state.global_step),
            expected_max_steps=int(trainer.state.max_steps),
            best_checkpoint_exists=bool(best),
            metrics_finite=metrics_are_finite_values(*[v for k, v in metrics.items() if isinstance(v, (int, float))]),
            frozen_test_accessed=frozen_flag,
            started_from_base_or_same_experiment=started_ok,
        )
        # Persist lightweight training history pointer
        history_path = PATHS["results"] / "training_history.json"
        history_path.write_text(
            json.dumps({"log_history": list(getattr(trainer.state, "log_history", []) or []), "note": "monitor metrics only"}, indent=2),
            encoding="utf-8",
        )
        payload = {
            "status": status,
            "contract_hash": mt_contract["contract_hash"],
            "mt_data_contract_hash": mt_contract["contract_hash"],
            "mt_training_contract_hash": mt_train_contract["mt_training_contract_hash"],
            "experiment_id": EXPERIMENT_ID,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "tokenizer_fingerprint": tok_fp,
            "global_step": int(trainer.state.global_step),
            "best_checkpoint": str(best) if best else None,
            "latest_checkpoint": str(latest_path) if latest_path else None,
            "best_checkpoint_name": best_name,
            "latest_checkpoint_name": latest_name,
            "metrics_monitor": metrics,
            "metrics_note": "eval_* during training is MONITOR subset SacreBLEU/chrF++; final full-validation metrics come from FULL_STAGE=evaluate",
            "monitor_size": MT_MONITOR_SIZE,
            "monitor_uid_set_hash": monitor_manifest["uid_set_hash"],
            "hparams": hparams,
            "generation_config": gen_cfg,
            "train_eligible_count": int(len(train_elig)),
            "validation_eligible_count": int(len(val_elig)),
            "started_from_base_or_same_experiment": started_ok,
            "frozen_test_accessed": frozen_flag,
        }
        write_mt_train_summary(FULL_STATE_DIR, payload)
        (PATHS["results"] / "mt_train_summary.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
        # checkpoint pointers (no weights)
        (PATHS["results"] / "best_checkpoint_pointer.json").write_text(
            json.dumps({"best_checkpoint": str(best), "best_checkpoint_name": best_name}, indent=2), encoding="utf-8"
        )
        (PATHS["results"] / "latest_checkpoint_pointer.json").write_text(
            json.dumps({"latest_checkpoint": str(latest_path), "latest_checkpoint_name": latest_name}, indent=2), encoding="utf-8"
        )
        full_status = status
        print(json.dumps(payload, indent=2, default=str))
        if status != STATUS_MT_TRAINING:
            raise RuntimeError("MT training status not SUCCESS")
    except Exception as e:
        fail_stage("mt_train", e)


In [ ]:
# FULL_STAGE=evaluate — fresh model load, full validation
if can_continue and RUN_MODE == "full" and FULL_STAGE == "evaluate":
    try:
        assert_locked_bartpho_baseline(MODEL_ID, MODEL_REVISION)
        assert_ready_for_mt_evaluate(FULL_STATE_DIR, contract=mt_contract)
        assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
        val_elig = pd.read_csv(FULL_STATE_DIR / "mt_validation_eligible.csv")
        LOADED_SPLITS.append("validation")
        tokenizer = load_mt_tokenizer(MODEL_ID, MODEL_REVISION)
        tok_fp_eval = tokenizer_fingerprint(tokenizer)
        train_summary = load_mt_train_success(FULL_STATE_DIR, expected_contract_hash=mt_contract["contract_hash"])
        training_contract = load_mt_training_contract(FULL_STATE_DIR)

        # Rebuild current training contract and compare
        monitor_manifest = json.loads((FULL_STATE_DIR / "mt_validation_monitor_manifest.json").read_text(encoding="utf-8"))
        current_train_contract = build_mt_training_contract(
            mt_data_contract_hash=str(mt_contract["contract_hash"]),
            train_uid_set_hash=str(mt_contract["train_uid_set_hash"]),
            validation_uid_set_hash=str(mt_contract["validation_uid_set_hash"]),
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp_eval,
            experiment_id=EXPERIMENT_ID,
            seed=SEED,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            max_steps=MAX_STEPS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            fp16=FP16,
            bf16=BF16,
            gradient_checkpointing=GRADIENT_CHECKPOINTING,
            save_steps=SAVE_STEPS,
            eval_steps=EVAL_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            max_source_length=MAX_SOURCE_LENGTH,
            max_target_length=MAX_TARGET_LENGTH,
            generation_max_length=GENERATION_MAX_LENGTH,
            num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL,
            greater_is_better=GREATER_IS_BETTER,
            monitor_size=MT_MONITOR_SIZE,
            monitor_uid_set_hash=str(monitor_manifest["uid_set_hash"]),
        )
        gen_cfg = build_generation_config(
            model_id=MODEL_ID, model_revision=MODEL_REVISION, tokenizer_fingerprint=tok_fp_eval,
            max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH,
            generation_max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            metric_for_best_model=METRIC_FOR_BEST_MODEL, greater_is_better=GREATER_IS_BETTER,
            normalization_version=mt_normalization_version(),
        )
        assert_evaluate_config_consistency(
            train_summary=train_summary,
            training_contract=training_contract,
            current_training_contract=current_train_contract,
            generation_config=gen_cfg,
            model_id=MODEL_ID,
            model_revision=MODEL_REVISION,
            tokenizer_fingerprint=tok_fp_eval,
        )

        local_exp = mt_experiment_dir(LOCAL_CKPT_ROOT, kind=FULL_TRAIN_MARKER, experiment_id=EXPERIMENT_ID)
        best = resolve_mt_best_checkpoint_for_evaluate(
            full_state_dir=FULL_STATE_DIR,
            experiment_id=EXPERIMENT_ID,
            train_summary=train_summary,
            local_experiment_dir=local_exp,
            expected_training_contract=training_contract,
        )
        from transformers import AutoModelForSeq2SeqLM
        model = AutoModelForSeq2SeqLM.from_pretrained(str(best))
        model.to(device)
        ds = MtTextDataset(val_elig, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
        pred_rows = generate_mt_predictions(
            model=model, tokenizer=tokenizer, dataset=ds,
            generation_max_length=GENERATION_MAX_LENGTH, num_beams=NUM_BEAMS,
            batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        )
        metrics = evaluate_mt_predictions(pred_rows)
        assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
        frozen_flag = derive_frozen_test_accessed(OPENED_PATHS, LOADED_SPLITS)
        status = derive_mt_evaluate_status(
            prediction_count=len(pred_rows),
            validation_count=len(val_elig),
            metrics_finite=bool(metrics.get("finite")) and metrics_are_finite_values(metrics.get("sacrebleu"), metrics.get("chrfpp")),
            frozen_test_accessed=frozen_flag,
            contract_ok=True,
        )
        pred_path = PATHS["results"] / "mt_validation_predictions.csv"
        pd.DataFrame(pred_rows).to_csv(pred_path, index=False)
        payload = {
            "status": status,
            "contract_hash": mt_contract["contract_hash"],
            "mt_data_contract_hash": mt_contract["contract_hash"],
            "mt_training_contract_hash": training_contract["mt_training_contract_hash"],
            "experiment_id": EXPERIMENT_ID,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "tokenizer_fingerprint": tok_fp_eval,
            "normalization_version": mt_normalization_version(),
            "generation_config": gen_cfg,
            "best_checkpoint": str(best),
            "n_validation": int(len(val_elig)),
            "prediction_count": int(len(pred_rows)),
            "sacrebleu": metrics["sacrebleu"],
            "chrfpp": metrics["chrfpp"],
            "sacrebleu_version": metrics.get("sacrebleu_version"),
            "sacrebleu_signature": metrics.get("sacrebleu_signature"),
            "chrf_signature": metrics.get("chrf_signature"),
            "metrics_scope": "full_validation_eligible",
            "monitor_sacrebleu_note": "Training best-ckpt used monitor subset; this score is full validation.",
            "frozen_test_accessed": frozen_flag,
            "stage_version": "mt_evaluate_v1",
        }
        write_mt_evaluate_summary(FULL_STATE_DIR, payload)
        (PATHS["results"] / "mt_evaluation_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
        full_status = status
        print(json.dumps(payload, indent=2))
        if status != STATUS_MT_EVALUATE:
            raise RuntimeError("MT evaluate failed gates")
    except Exception as e:
        fail_stage("mt_evaluate", e)


## Final status (stage-aware)

In [ ]:
# Stage-aware status — do not dump pilot check lists in full mode
if pipeline_error is not None:
    STATUS = STATUS_FAILED
elif full_status:
    STATUS = full_status
else:
    STATUS = STATUS_FAILED if RUN_MODE == "full" else "SUCCESS_MT_PILOT_SKIPPED"

assert_no_frozen_test_access(OPENED_PATHS, LOADED_SPLITS)
frozen_flag = derive_frozen_test_accessed(OPENED_PATHS, LOADED_SPLITS)
hand = derive_notebook04_handoff(STATUS, frozen_test_accessed=frozen_flag)

run_summary = {
    "run_id": RUN_ID,
    "status": STATUS,
    "run_mode": RUN_MODE,
    "full_stage": FULL_STAGE,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "tokenizer_fingerprint": tok_fp,
    "contract_hash": (mt_contract or {}).get("contract_hash"),
    "experiment_id": EXPERIMENT_ID,
    "pipeline_error_stage": pipeline_error_stage,
    "pipeline_error": pipeline_error,
    "opened_paths": OPENED_PATHS,
    "loaded_splits": LOADED_SPLITS,
    **hand,
}
(PATHS["results"] / "run_summary.json").write_text(json.dumps(run_summary, indent=2, default=str), encoding="utf-8")
print("=" * 60)
print("NOTEBOOK 04 STATUS:", STATUS)
print("RUN MODE:", RUN_MODE, " FULL_STAGE:", FULL_STAGE)
print("READY FOR CASCADED C0:", hand["ready_for_cascaded_c0"])
print("READY FOR RQ1 FINAL:", hand["ready_for_rq1_final"])
print("FROZEN TEST ACCESSED:", hand["frozen_test_accessed"])
print("=" * 60)
if STATUS == STATUS_FAILED:
    raise RuntimeError(f"NOTEBOOK 04 FAILED stage={pipeline_error_stage} reason={pipeline_error}")


In [ ]:
# Export
def _pkg_ver(name: str):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", None)
    except Exception:
        return None

_env = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": getattr(torch, "__version__", None),
    "torch_cuda_version": getattr(getattr(torch, "version", None), "cuda", None),
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    "transformers": _pkg_ver("transformers"),
    "accelerate": _pkg_ver("accelerate"),
    "sacrebleu": _pkg_ver("sacrebleu"),
    "sentencepiece": _pkg_ver("sentencepiece"),
    "BAHNAR_LOCAL_ROOT": os.environ.get("BAHNAR_LOCAL_ROOT"),
    "BAHNAR_DURABLE_ROOT": os.environ.get("BAHNAR_DURABLE_ROOT"),
    "BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES": os.environ.get("BAHNAR_DURABLE_CHECKPOINT_BUDGET_BYTES"),
}
(PATHS["results"] / "environment.json").write_text(
    json.dumps(_env, indent=2, default=str), encoding="utf-8"
)

export_dir = export_notebook04_run(
    export_root=RUNTIME.export_root,
    run_id=RUN_ID,
    artifacts_dir=PATHS["artifacts"],
    results_dir=PATHS["results"],
    full_state_dir=FULL_STATE_DIR if isinstance(FULL_STATE_DIR, Path) else Path(str(FULL_STATE_DIR)),
    run_config={
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "full_stage": FULL_STAGE,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "experiment_id": EXPERIMENT_ID,
        "metric_for_best_model": METRIC_FOR_BEST_MODEL,
        "generation_max_length": GENERATION_MAX_LENGTH,
        "num_beams": NUM_BEAMS,
        "mt_monitor_size": MT_MONITOR_SIZE,
        "hard_max_unk_rate": HARD_MAX_UNK_RATE,
        "hard_max_truncation_rate": HARD_MAX_TRUNCATION_RATE,
    },
    environment=_env,
    pointer={
        "run_id": RUN_ID,
        "full_stage": FULL_STAGE,
        "experiment_id": EXPERIMENT_ID,
        "status": STATUS,
        "full_state_dir": str(FULL_STATE_DIR) if FULL_STATE_DIR else None,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    },
)
print("Exported:", export_dir)
print("Environment:", json.dumps({k: _env[k] for k in ("python", "torch", "transformers", "gpu_name")}, default=str))
